In [1]:
import asyncio
from random import random

In [2]:
async def echo(index: int):
    await asyncio.sleep(0.1)
    return index


async def echo_random_latency(index: int):
    await asyncio.sleep(random())
    return index

# Test Executor 

_**NOTE**: Requires `ipywidgets` installed_

In [ ]:
from ragas.async_utils import is_event_loop_running, as_completed

In [4]:
assert is_event_loop_running() is True, "is_event_loop_running() returned False"

In [5]:
async def _run():
    results = []
    for t in as_completed([echo(1), echo(2), echo(3)], 3):
        r = await t
        results.append(r)
    return results


results = await _run()

expected = [1, 2, 3]
assert results == expected, f"got: {results}, expected: {expected}"

## Test Executor

In [6]:
from ragas.executor import Executor

In [7]:
try:
    # test order of results when they should return in submission order
    executor = Executor(raise_exceptions=True)
    for i in range(10):
        executor.submit(echo, i, name=f"echo_{i}")

    _ = executor.results()

except RuntimeError as e:
    assert str(e) == "In Jupyter/IPython, use `await executor.aresults()` directly; this avoids the need for nest_asyncio."

In [11]:
# test order of results when they should return in submission order
executor = Executor(raise_exceptions=True)
for i in range(10):
    executor.submit(echo, i, name=f"echo_{i}")

results = await executor.aresults()
assert results == list(range(10))


Evaluating:   0%|          | 0/10 [00:00<?, ?it/s]

In [12]:
# test order of results when may return unordered
executor = Executor(batch_size=None)

# add jobs to the executor
for i in range(10):
    executor.submit(echo_random_latency, i, name=f"echo_order_{i}")

# Act
results = await executor.aresults()
# Assert
assert results == list(range(10))

Evaluating:   0%|          | 0/10 [00:00<?, ?it/s]

In [14]:
# Test output order; batching
executor = Executor(batch_size=3)

# add jobs to the executor
for i in range(10):
    executor.submit(echo_random_latency, i, name=f"echo_order_{i}")

# Act
results = await executor.aresults()
# Assert
assert results == list(range(10))

Evaluating:   0%|          | 0/10 [00:00<?, ?it/s]

Batch 1/4:   0%|          | 0/3 [00:00<?, ?it/s]

In [15]:
# Test no progress
executor = Executor(show_progress=False)

# add jobs to the executor
for i in range(10):
    executor.submit(echo_random_latency, i, name=f"echo_order_{i}")

# Act
results = await executor.aresults()
# Assert
assert results == list(range(10))

In [17]:
# Test multiple submission sets
executor = Executor(raise_exceptions=True)
for i in range(1000):
    executor.submit(asyncio.sleep, 0.01)

results = await executor.aresults()
assert results, "Results should be list of None"

for i in range(1000):
    executor.submit(asyncio.sleep, 0.01)

results = await executor.aresults()
assert results, "Results should be list of None"

Evaluating:   0%|          | 0/1000 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/1000 [00:00<?, ?it/s]

# Test Metric

In [18]:
from ragas.metrics.base import Metric


class FakeMetric(Metric):
    name = "fake_metric"
    _required_columns = ("user_input", "response")

    def init(self):
        pass

    async def _ascore(self, row, callbacks) -> float:
        return 0


fm = FakeMetric()

In [19]:
score = fm.score({"user_input": "a", "response": "b"})
assert score == 0

/var/folders/7l/wjd9bkjx0c1dmb1wjpzj_3zc0000gp/T/ipykernel_79567/3943354553.py:1: DeprecationWarning: The function score was deprecated in 0.2, and will be removed in the 0.3 release. Use single_turn_ascore instead.
  score = fm.score({"user_input": "a", "response": "b"})


# Test run_async_tasks

In [20]:
from ragas.async_utils import run_async_tasks

In [21]:
# run tasks unbatched
tasks = [echo_random_latency(i) for i in range(10)]
results = run_async_tasks(tasks, batch_size=None, show_progress=True)
# Assert
assert sorted(results) == list(range(10))

Running async tasks:   0%|          | 0/10 [00:00<?, ?it/s]

In [22]:
# run tasks batched
tasks = [echo_random_latency(i) for i in range(10)]
results = run_async_tasks(tasks, batch_size=3, show_progress=True)
# Assert
assert sorted(results) == list(range(10))

Running async tasks:   0%|          | 0/10 [00:00<?, ?it/s]

Batch 1/4:   0%|          | 0/3 [00:00<?, ?it/s]

In [23]:
# Test no progress
tasks = [echo_random_latency(i) for i in range(10)]
results = run_async_tasks(tasks, batch_size=3, show_progress=False)
# Assert
assert sorted(results) == list(range(10))